# EDA — Pay Equity Analysis
Exploring the HR dataset to understand salary distribution and gender pay gaps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/processed/hr_clean.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 1. Dataset Overview

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print(f"\n=== Null Values ===")
print(df.isnull().sum())
print(f"\n=== Duplicates: {df.duplicated().sum()}")
print(f"\n=== Basic Stats ===")
df.describe().round(2)

## 2. Salary Distribution

In [ ]:
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.labelcolor': 'white', 'xtick.color': 'white', 'ytick.color': 'white',
    'text.color': 'white', 'axes.titlecolor': 'white', 'axes.edgecolor': '#333355', 'grid.color': '#333355',
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

axes[0].hist(df['MonthlyIncome'], bins=40, color='#7B5EA7', edgecolor='#0f1117', alpha=0.9)
axes[0].axvline(df['MonthlyIncome'].mean(), color='#E07B54', lw=2, linestyle='--', label=f"Mean: ${df['MonthlyIncome'].mean():,.0f}")
axes[0].axvline(df['MonthlyIncome'].median(), color='#54A0E0', lw=2, linestyle='--', label=f"Median: ${df['MonthlyIncome'].median():,.0f}")
axes[0].set_title('Monthly Income Distribution', fontweight='bold')
axes[0].set_xlabel('Monthly Income ($)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

gender_income = df.groupby('Gender')['MonthlyIncome'].mean()
gap_pct = ((gender_income['Male'] - gender_income['Female']) / gender_income['Male']) * 100
bars = axes[1].bar(gender_income.index, gender_income.values, color=['#E07B54', '#54A0E0'], edgecolor='#0f1117', width=0.5)
for bar, val in zip(bars, gender_income.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f'${val:,.0f}', ha='center', color='white', fontweight='bold')
axes[1].set_title(f'Avg Income by Gender (Pay Gap: {gap_pct:.1f}%)', fontweight='bold')
axes[1].set_ylabel('Average Monthly Income ($)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../reports/fig1_pay_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f"\n📌 Key Finding: Males earn ${gender_income['Male']:,.0f}/mo vs Females ${gender_income['Female']:,.0f}/mo — a {gap_pct:.1f}% pay gap.")

## 3. Pay Gap by Department & Job Level

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

dept_gender = df.groupby(['Department', 'Gender'])['MonthlyIncome'].mean().unstack()
dept_gender.plot(kind='bar', ax=axes[0], color=['#E07B54', '#54A0E0'], edgecolor='#0f1117', width=0.7)
axes[0].set_title('Avg Income by Department & Gender', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Avg Monthly Income ($)')
axes[0].legend(title='Gender', facecolor='#1a1d2e', labelcolor='white')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, alpha=0.3, axis='y')

jl_gender = df.groupby(['JobLevel', 'Gender'])['MonthlyIncome'].mean().unstack()
jl_labels = {1:'Junior', 2:'Mid', 3:'Senior', 4:'Lead', 5:'Executive'}
jl_gender.index = [jl_labels[i] for i in jl_gender.index]
jl_gender.plot(kind='bar', ax=axes[1], color=['#E07B54', '#54A0E0'], edgecolor='#0f1117', width=0.7)
axes[1].set_title('Pay Gap Across Job Levels', fontweight='bold')
axes[1].set_xlabel('Job Level')
axes[1].set_ylabel('Avg Monthly Income ($)')
axes[1].legend(title='Gender', facecolor='#1a1d2e', labelcolor='white')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../reports/fig2_pay_gap_breakdown.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print("Department avg salaries:")
print(df.groupby('Department')['MonthlyIncome'].mean().round(0))

## 4. Salary vs Experience & Pay Gap by Role

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

for gender, color in [('Male', '#54A0E0'), ('Female', '#E07B54')]:
    subset = df[df['Gender'] == gender]
    axes[0].scatter(subset['TotalWorkingYears'], subset['MonthlyIncome'], alpha=0.4, s=15, c=color, label=gender)
axes[0].set_title('Income vs Experience by Gender', fontweight='bold')
axes[0].set_xlabel('Total Working Years')
axes[0].set_ylabel('Monthly Income ($)')
axes[0].legend(facecolor='#1a1d2e', labelcolor='white')
axes[0].grid(True, alpha=0.3)

role_gap = df.groupby(['JobRole', 'Gender'])['MonthlyIncome'].mean().unstack()
role_gap['gap_pct'] = ((role_gap['Male'] - role_gap['Female']) / role_gap['Male'] * 100).fillna(0)
role_gap = role_gap.sort_values('gap_pct', ascending=True)
bar_colors = ['#5EC87B' if x < 5 else '#E07B54' if x > 10 else '#E0C454' for x in role_gap['gap_pct']]
axes[1].barh(role_gap.index, role_gap['gap_pct'], color=bar_colors, edgecolor='#0f1117')
axes[1].set_title('Gender Pay Gap % by Job Role', fontweight='bold')
axes[1].set_xlabel('Pay Gap (%)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../reports/fig3_salary_experience.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 5. Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d2e')

num_cols = ['Age', 'Education', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome',
            'PerformanceRating', 'TotalWorkingYears', 'WorkLifeBalance', 'YearsAtCompany']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(250, 30, l=65, center='dark', as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            square=True, linewidths=0.5, annot=True, fmt='.2f',
            annot_kws={'size': 9}, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('../reports/fig4_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print("Top correlates with MonthlyIncome:")
print(corr['MonthlyIncome'].sort_values(ascending=False).round(3))

## Summary

- Male avg: $15,810 | Female avg: $14,351 | Gap: 9.2%
- Job Level is the strongest salary driver (r=0.85)
- HR dept is the lowest paid across all departments